# 2025-26 Case Mining — Eval Queues + Slang Candidates

**Objective:** Deterministically mine real comments to expand the classifier eval
suite (#62 items 1+2): labeling queues for four target categories, plus a ranked
slang-candidate table for the item-2 audit. Deliverable: this notebook + queue
YAMLs and derived parquets under `data/2025-26/reference/eval_mining/`. Labeled
survivors merge into `tests/eval/cases.yaml`. **No prompt changes happen here** —
slang/sarcasm prompt-line decisions belong to the experiment PR.

**This notebook answers:**

- §1 — What does the v2 candidate pool look like (short / multi-player / `/s`
  volumes), and are the selection thresholds right?
- §2–§4 — v2 queues: short comments (1–3 words), multi-player comments,
  `/s`-stripped sarcasm.
- §5 — v1 queue: genuine praise of struggling players (the regression trap the
  sarcasm-line experiment must not trip).
- §6 — Which terms lean pos/neg in v1 classified data, and what is new in 25-26
  (slang candidates for nba-superfan verification)?
- §7 — Verdict: queue counts, determinism hashes, handoff checklist.

## Load Data

Guardrails (per 02/03 house pattern):

- **`body` is projected in exactly three places**: the §1 pool scan (candidate
  bodies *are* the deliverable — the pool parquet stays small), the §5 v1
  genuine-praise selection, and the §6 tokenization passes (only grouped
  aggregates and capped snippets leave DuckDB). Bodies are never dumped
  wholesale; queues cap at `N_PER_CATEGORY`.
- Every scan is guarded on artifact existence with a staleness warning if the
  artifact predates its source file.
- **Determinism:** all sampling is `ORDER BY md5(id)` — stable across DuckDB
  versions and re-runs; never `hash()` or `random()`. Re-running this notebook
  against the frozen inputs (v2 population is final at 2.15M; v1 is immutable)
  reproduces the queues byte-for-byte.
- Queues carry `proposed`, not `expected`: pasting an unlabeled candidate into
  `cases.yaml` fails `load_cases()` loudly ("missing keys: ['expected']").

In [1]:
import hashlib
import os
import re
import sys
from pathlib import Path

# Bootstrap: run from anywhere (see 01/02/03 — walk up to the repo root, chdir).
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import duckdb
import polars as pl
import yaml

from pipeline.evaluation import load_cases
from utils.paths import get_data_dir
from utils.season_config import get_active_season

SEASON = "2025-26"  # pinned: this notebook is season-scoped (notebooks/2025-26/)
assert get_active_season() == SEASON, "active season flipped — review before re-running"

MENTIONS_RAW = get_data_dir(season=SEASON) / "filtered" / "r_nba_player_mentions.jsonl"
COVERAGE_DIR = get_data_dir(season=SEASON) / "reference" / "coverage"
MINING_DIR = get_data_dir(season=SEASON) / "reference" / "eval_mining"
MINING_DIR.mkdir(parents=True, exist_ok=True)

V1_SENTIMENT = get_data_dir(season="2024-25") / "processed" / "sentiment.parquet"
V1_PLAYER_OVERALL = get_data_dir(season="2024-25") / "dashboard" / "player_overall.parquet"

assert MENTIONS_RAW.exists(), f"missing mentions file: {MENTIONS_RAW}"
assert V1_SENTIMENT.exists(), f"missing v1 sentiment parquet: {V1_SENTIMENT}"

# Existing eval cases: dedup targets and id-numbering context.
EXISTING_CASES = load_cases()
SEEN_TEXTS = {c.text.strip().lower() for c in EXISTING_CASES}
SEEN_IDS = {c.comment_id for c in EXISTING_CASES if c.comment_id}

N_PER_CATEGORY = 30


def warn_if_stale(pq: Path, raw: Path) -> None:
    """Companion to the existence-only reuse guards: flag a persisted
    artifact older than the file it derives from."""
    if pq.exists() and pq.stat().st_mtime < raw.stat().st_mtime:
        print(f"WARNING: {pq.name} predates {raw.name} — delete it to force a rescan")


def sql_quote(s: str) -> str:
    """Escape a string for embedding in a single-quoted SQL literal."""
    return s.replace("'", "''")


# /s marker: requires a non-word, non-slash char (or string boundary) on both
# sides — catches "... /s", "wow /s\nanyway"; rejects "b/s" (word char before)
# and URL path segments like "reddit.com/r/nba/s/abc" (word chars both sides).
S_TAG_SQL = r"(^|[^\w/])/s([^\w/]|$)"
S_TAG_RE = re.compile(S_TAG_SQL)


def strip_s_tag(text: str) -> str | None:
    """Remove the /s marker, collapsing leftover runs of spaces.

    Returns None when nothing was stripped or nothing remains — callers skip
    those rows."""
    stripped = S_TAG_RE.sub(r"\1\2", text)
    stripped = re.sub(r"[ \t]{2,}", " ", stripped).strip()
    if not stripped or stripped == text.strip():
        return None
    return stripped


# Regex self-test — the shapes that must (not) match.
assert S_TAG_RE.search("Great defense as usual /s")
assert S_TAG_RE.search("elite shooting /s\nanyway")
assert not S_TAG_RE.search("that call was total b/s honestly")
assert not S_TAG_RE.search("see reddit.com/r/nba/s/abc123")
assert strip_s_tag("Great defense as usual /s") == "Great defense as usual"
assert strip_s_tag("no tag here") is None


def dedup_take(df: pl.DataFrame, n: int) -> list[dict]:
    """First n rows not colliding with existing cases or earlier queues.

    Mutates the shared SEEN sets, so cross-queue dedup falls out of section
    order (§4 sarcasm > §3 multi > §2 short by construction of the pool
    predicates; §5 uses v1 ids)."""
    out: list[dict] = []
    for r in df.iter_rows(named=True):
        key = r["body"].strip().lower()
        if key in SEEN_TEXTS or r["id"] in SEEN_IDS:
            continue
        SEEN_TEXTS.add(key)
        SEEN_IDS.add(r["id"])
        out.append(r)
        if len(out) == n:
            break
    return out


def make_candidate(
    cid: str,
    text: str,
    proposed: str | None,
    proposed_player: str | None,
    category: str,
    source: str,
    comment_id: str,
    note: str | None,
    context: dict,
) -> dict:
    """Queue entry in the documented field order."""
    return {
        "id": cid,
        "text": text,
        "proposed": proposed,
        "proposed_player": proposed_player,
        "category": category,
        "source": source,
        "comment_id": comment_id,
        "note": note,
        "context": context,
    }


def queue_header(filename: str, section: str) -> str:
    return (
        f"# {filename} — written by notebooks/2025-26/05_case_mining.ipynb ({section})\n"
        "#\n"
        "# LABELING:\n"
        "#   1. Decide each candidate: fix `proposed` (pos|neg|neu), then RENAME\n"
        "#      proposed -> expected and proposed_player -> expected_player.\n"
        "#   2. Delete rejects entirely; delete the `context:` block from keepers\n"
        "#      (fold anything worth keeping into `note`).\n"
        "#   3. Copy survivors into tests/eval/cases.yaml under their category.\n"
        "#      New categories need a meta.category_floors entry (placeholder 0.0).\n"
        "#   Unrenamed entries fail load_cases() by design (missing `expected`).\n"
        "#\n"
    )


def write_queue(path: Path, rows: list[dict], section: str) -> None:
    body = yaml.safe_dump(
        {"candidates": rows}, sort_keys=False, allow_unicode=True, width=4096
    )
    path.write_text(queue_header(path.name, section) + body)
    print(f"wrote {len(rows)} candidates -> {path}")


duckdb.sql("SET temp_directory = '/tmp/duckdb_mining_spill'")

print(f"Mentions file: {MENTIONS_RAW}  ({MENTIONS_RAW.stat().st_size / 1e9:.1f} GB)")
print(f"v1 sentiment: {V1_SENTIMENT}  ({V1_SENTIMENT.stat().st_size / 1e6:.0f} MB)")
print(f"Existing eval cases: {len(EXISTING_CASES)}  (dedup sets primed)")
print(f"Queue size target: {N_PER_CATEGORY} per category")

Mentions file: data/2025-26/filtered/r_nba_player_mentions.jsonl  (1.0 GB)
v1 sentiment: data/2024-25/processed/sentiment.parquet  (169 MB)
Existing eval cases: 23  (dedup sets primed)
Queue size target: 30 per category


## 1. v2 candidate pool — one pass, then tuning stats

The ONE body-projecting pass over the 1 GB mentions file: rows matching *any*
target predicate (≤3 words, multi-player, `/s`-tagged) survive to a small pool
parquet with hygiene filters applied; every queue section below selects from the
pool, rescan-free. `length(body) <= 300` is applied at selection time (short
bodies are short by definition; it would be a no-op there).

In [2]:
# §1 scan — the ONE v2 body-projecting pass (plus §6b's aggregate-only pass).
POOL_PQ = MINING_DIR / "v2_candidates_pool.parquet"
warn_if_stale(POOL_PQ, MENTIONS_RAW)

if not POOL_PQ.exists():
    duckdb.sql(f"""
        COPY (
            SELECT
                id, body, score, author_flair_text, mentioned_players,
                length(regexp_split_to_array(trim(body), '\s+')) AS n_words,
                len(mentioned_players)                           AS n_players,
                regexp_matches(body, '{S_TAG_SQL}')              AS has_s_tag
            FROM read_ndjson('{MENTIONS_RAW}', format='newline_delimited',
                 columns={{id: 'VARCHAR', body: 'VARCHAR', score: 'BIGINT',
                           author: 'VARCHAR', author_flair_text: 'VARCHAR',
                           mentioned_players: 'VARCHAR[]'}})
            WHERE body NOT IN ('[deleted]', '[removed]')
              AND author <> 'AutoModerator'
              AND NOT regexp_matches(trim(body), '^https?://\S+$')
              AND (
                       length(regexp_split_to_array(trim(body), '\s+')) <= 3
                    OR len(mentioned_players) > 1
                    OR regexp_matches(body, '{S_TAG_SQL}')
                  )
        ) TO '{POOL_PQ}' (FORMAT parquet)
    """)

pool_stats = duckdb.sql(f"""
    SELECT
        count(*)                                                    AS pool_rows,
        count(*) FILTER (n_words <= 3 AND n_players = 1
                         AND NOT has_s_tag)                         AS short_rows,
        count(*) FILTER (n_players > 1 AND NOT has_s_tag)           AS multi_rows,
        count(*) FILTER (has_s_tag)                                 AS s_tag_rows
    FROM read_parquet('{POOL_PQ}')
""").pl()
print(pool_stats)

shape: (1, 4)
┌───────────┬────────────┬────────────┬────────────┐
│ pool_rows ┆ short_rows ┆ multi_rows ┆ s_tag_rows │
│ ---       ┆ ---        ┆ ---        ┆ ---        │
│ i64       ┆ i64        ┆ i64        ┆ i64        │
╞═══════════╪════════════╪════════════╪════════════╡
│ 696865    ┆ 77487      ┆ 616951     ┆ 2427       │
└───────────┴────────────┴────────────┴────────────┘


In [3]:
# §1 tuning stats — eyeball before the selections below.
print("Short-comment word-count distribution (single-player, no /s):")
print(duckdb.sql(f"""
    SELECT n_words, count(*) AS n
    FROM read_parquet('{POOL_PQ}')
    WHERE n_words <= 3 AND n_players = 1 AND NOT has_s_tag
    GROUP BY n_words ORDER BY n_words
""").pl())

print("\n/s-tagged rows by word count (>=4 words feeds §4):")
print(duckdb.sql(f"""
    SELECT n_words >= 4 AS eligible, count(*) AS n
    FROM read_parquet('{POOL_PQ}') WHERE has_s_tag GROUP BY 1 ORDER BY 1
""").pl())

print("\nMulti-player split in the pool (corpus-wide view: 03's artifact):")
print(duckdb.sql(f"""
    SELECT n_players, count(*) AS n
    FROM read_parquet('{POOL_PQ}')
    WHERE n_players > 1 AND NOT has_s_tag
    GROUP BY n_players ORDER BY n_players LIMIT 6
""").pl())

_corpus_np = COVERAGE_DIR / "mentions_per_comment.parquet"
if _corpus_np.exists():
    print("\nCorpus-wide mentions-per-comment (from 03 §1):")
    print(pl.read_parquet(_corpus_np))

Short-comment word-count distribution (single-player, no /s):
shape: (3, 2)
┌─────────┬───────┐
│ n_words ┆ n     │
│ ---     ┆ ---   │
│ i64     ┆ i64   │
╞═════════╪═══════╡
│ 1       ┆ 10059 │
│ 2       ┆ 27578 │
│ 3       ┆ 39850 │
└─────────┴───────┘

/s-tagged rows by word count (>=4 words feeds §4):
shape: (2, 2)
┌──────────┬──────┐
│ eligible ┆ n    │
│ ---      ┆ ---  │
│ bool     ┆ i64  │
╞══════════╪══════╡
│ false    ┆ 34   │
│ true     ┆ 2393 │
└──────────┴──────┘

Multi-player split in the pool (corpus-wide view: 03's artifact):
shape: (6, 2)
┌───────────┬────────┐
│ n_players ┆ n      │
│ ---       ┆ ---    │
│ i64       ┆ i64    │
╞═══════════╪════════╡
│ 2         ┆ 422243 │
│ 3         ┆ 114090 │
│ 4         ┆ 42794  │
│ 5         ┆ 18185  │
│ 6         ┆ 8440   │
│ 7         ┆ 4339   │
└───────────┴────────┘

Corpus-wide mentions-per-comment (from 03 §1):
shape: (48, 5)
┌───────────┬─────────┬────────────────┬────────────┬────────────┐
│ n_players ┆ n_rows  ┆ n_disti

## 2. Short-comment queue (1–3 words)

Minimal-context classification is the stress; a mechanical `proposed` label
would anchor the labeler, so it stays null. Stratified ~10 per word-count bucket
so one-word comments aren't drowned out. Attribution is mechanical here
(single mentioned player), so `proposed_player` is filled.

In [4]:
# §2 — short queue.
short_rows: list[tuple[dict, int]] = []
for wc in (1, 2, 3):
    df = duckdb.sql(f"""
        SELECT id, body, score, author_flair_text, mentioned_players
        FROM read_parquet('{POOL_PQ}')
        WHERE n_words = {wc} AND n_players = 1 AND NOT has_s_tag
        ORDER BY md5(id)
        LIMIT {N_PER_CATEGORY}
    """).pl()
    short_rows += [(r, wc) for r in dedup_take(df, N_PER_CATEGORY // 3)]

short_queue = [
    make_candidate(
        cid=f"short-m{i:02d}",
        text=r["body"].strip(),
        proposed=None,
        proposed_player=r["mentioned_players"][0],
        category="short",
        source="mined-v2",
        comment_id=r["id"],
        note=None,
        context={
            "score": r["score"],
            "author_flair_text": r["author_flair_text"],
            "mentioned_players": list(r["mentioned_players"]),
            "n_words": wc,
        },
    )
    for i, (r, wc) in enumerate(short_rows, start=1)
]
write_queue(MINING_DIR / "candidates_short.yaml", short_queue, "§2")
for c in short_queue[:5]:
    print(f"  {c['id']}: {c['text']!r} -> {c['proposed_player']}")

wrote 30 candidates -> data/2025-26/reference/eval_mining/candidates_short.yaml
  short-m01: 'Harper!!!' -> Dylan Harper
  short-m02: 'Draymond' -> Draymond Green
  short-m03: 'Giannis' -> Giannis Antetokounmpo
  short-m04: 'KD' -> Kevin Durant
  short-m05: 'Amen' -> Amen Thompson


## 3. Multi-player queue

The category's real payload is per-case attribution (`expected_player`).
Stratified: ~20 two-player comments carrying a deterministic contrast cue
(" but ", " while ", " unlike ", " meanwhile " — proxies for
plausibly-opposite sentiment), ~5 two-player without, ~5 with 3+ players.
Labeling guidance: `expected` = the dominant sentiment and `expected_player`
its target; genuinely balanced comments get rejected.

In [5]:
# §3 — multi-player queue.
_CUE = ("body ILIKE '% but %' OR body ILIKE '% while %' "
        "OR body ILIKE '% unlike %' OR body ILIKE '% meanwhile %'")
_BASE = "n_players > 1 AND NOT has_s_tag AND length(body) <= 300 AND n_words >= 4"

multi_rows: list[dict] = []
for label, predicate, quota in [
    ("cue", f"n_players = 2 AND ({_CUE})", 20),
    ("nocue", f"n_players = 2 AND NOT ({_CUE})", 5),
    ("three_plus", "n_players >= 3", 5),
]:
    df = duckdb.sql(f"""
        SELECT id, body, score, author_flair_text, mentioned_players
        FROM read_parquet('{POOL_PQ}')
        WHERE {_BASE} AND {predicate}
        ORDER BY md5(id)
        LIMIT {quota * 3}
    """).pl()
    taken = dedup_take(df, quota)
    print(f"  stratum {label}: {len(taken)}/{quota}")
    multi_rows += taken

multi_queue = [
    make_candidate(
        cid=f"multi-m{i:02d}",
        text=r["body"].strip(),
        proposed=None,
        proposed_player=None,
        category="multi_player",
        source="mined-v2",
        comment_id=r["id"],
        note=None,
        context={
            "score": r["score"],
            "author_flair_text": r["author_flair_text"],
            "mentioned_players": list(r["mentioned_players"]),
        },
    )
    for i, r in enumerate(multi_rows, start=1)
]
write_queue(MINING_DIR / "candidates_multi_player.yaml", multi_queue, "§3")

  stratum cue: 20/20


  stratum nocue: 5/5
  stratum three_plus: 5/5
wrote 30 candidates -> data/2025-26/reference/eval_mining/candidates_multi_player.yaml


## 4. Sarcasm queue (`/s` stripped)

`/s`-tagged comments are self-labeled sarcasm; stripping the tag yields
realistic *unmarked* sarcasm — the known accuracy ceiling — with near-ground-truth
labels. `proposed: neg` is a weak prior (the tag on player comments is
overwhelmingly mock praise), not a determination; the original tagged body is
kept in `context` for the labeler. Minimum 4 words to avoid double-dipping §2.

In [6]:
# §4 — sarcasm queue.
_s_df = duckdb.sql(f"""
    SELECT id, body, score, author_flair_text, mentioned_players
    FROM read_parquet('{POOL_PQ}')
    WHERE has_s_tag AND n_words >= 4 AND length(body) <= 300
    ORDER BY md5(id)
    LIMIT {N_PER_CATEGORY * 4}
""").pl()

sarcasm_rows: list[tuple[dict, str]] = []
for r in _s_df.iter_rows(named=True):
    stripped = strip_s_tag(r["body"])
    if stripped is None:
        continue
    key = stripped.lower()
    if key in SEEN_TEXTS or r["id"] in SEEN_IDS:
        continue
    SEEN_TEXTS.add(key)
    SEEN_IDS.add(r["id"])
    sarcasm_rows.append((r, stripped))
    if len(sarcasm_rows) == N_PER_CATEGORY:
        break

sarcasm_queue = [
    make_candidate(
        cid=f"sarcasm-m{i:02d}",
        text=stripped,
        proposed="neg",
        proposed_player=(
            r["mentioned_players"][0] if len(r["mentioned_players"]) == 1 else None
        ),
        category="sarcasm",
        source="mined-v2",
        comment_id=r["id"],
        note="/s tag stripped from source comment (05_case_mining §4); "
        "proposed label is a prior, not a determination",
        context={
            "score": r["score"],
            "author_flair_text": r["author_flair_text"],
            "mentioned_players": list(r["mentioned_players"]),
            "original_body": r["body"],
        },
    )
    for i, (r, stripped) in enumerate(sarcasm_rows, start=1)
]
write_queue(MINING_DIR / "candidates_sarcasm.yaml", sarcasm_queue, "§4")
for c in sarcasm_queue[:5]:
    print(f"  {c['id']}: {c['text'][:70]!r}")

wrote 30 candidates -> data/2025-26/reference/eval_mining/candidates_sarcasm.yaml
  sarcasm-m01: 'His Slovenian brother Luka Doncic introduced him to some texas bbq'
  sarcasm-m02: 'Out of uniform, Brunson definitely doesn’t look big enough to be an NB'
  sarcasm-m03: 'Yeah how dare that guy. \n\nThunder... Harden was the victim. \nRockets..'
  sarcasm-m04: 'I was getting ready to hate the guy wearing the Tatum jersey.'
  sarcasm-m05: "He's still anti-Giannis after all these years"


## 5. Genuine praise of struggling players (v1)

The regression trap: any sarcasm-prompt fix must not turn real praise of
high-neg-rate players into false negatives. Selection is *by the v1
classifier's own high-confidence pos label*, so `proposed` is mechanical here;
the labeler's job is rejecting the false positives (which belong to a different
category). Threshold: official-ranking volume bar (≥ 5,000 comments) and
`neg_rate >= 0.43` — tune from the table below.

`sentiment_player` in the v1 parquet is *raw classifier output*; it is matched
against the struggling players' full alias sets (period-stripped, lowercased),
mirroring `resolve_sentiment_player`.

In [7]:
# §5a — the struggling-player cut.
NEG_RATE_CUT = 0.43

overall = duckdb.sql(f"""
    SELECT attributed_player, neg_rate, comment_count
    FROM read_parquet('{V1_PLAYER_OVERALL}')
    WHERE comment_count >= 5000
    ORDER BY neg_rate DESC
    LIMIT 15
""").pl().with_columns((pl.col("neg_rate") >= NEG_RATE_CUT).alias("in_cut"))
print(overall)

STRUGGLING = overall.filter(pl.col("in_cut"))["attributed_player"].to_list()
print(f"\nSTRUGGLING (neg_rate >= {NEG_RATE_CUT}): {STRUGGLING}")

# Alias keys for the raw sentiment_player match — v1 config, both canonical
# names and aliases, lowercased and period-stripped.
with open("config/2024-25/players.yaml") as f:
    _v1_players = yaml.safe_load(f)["players"]


def _alias_keys(name: str) -> set[str]:
    entry = _v1_players.get(name) or {}
    aliases = entry.get("aliases", []) if isinstance(entry, dict) else list(entry)
    return {a.lower().replace(".", "").strip() for a in [name, *aliases]}


KEY_TO_CANON = {
    key: canon for canon in STRUGGLING for key in _alias_keys(canon)
}
print(f"alias keys: {len(KEY_TO_CANON)} across {len(STRUGGLING)} players")

shape: (15, 4)
┌───────────────────┬──────────┬───────────────┬────────┐
│ attributed_player ┆ neg_rate ┆ comment_count ┆ in_cut │
│ ---               ┆ ---      ┆ ---           ┆ ---    │
│ str               ┆ f64      ┆ i64           ┆ bool   │
╞═══════════════════╪══════════╪═══════════════╪════════╡
│ Draymond Green    ┆ 0.5105   ┆ 53454         ┆ true   │
│ Joel Embiid       ┆ 0.4927   ┆ 31538         ┆ true   │
│ Ben Simmons       ┆ 0.4561   ┆ 11123         ┆ true   │
│ Russell Westbrook ┆ 0.452    ┆ 40571         ┆ true   │
│ James Harden      ┆ 0.441    ┆ 28504         ┆ true   │
│ …                 ┆ …        ┆ …             ┆ …      │
│ Damian Lillard    ┆ 0.396    ┆ 12600         ┆ false  │
│ Julius Randle     ┆ 0.3958   ┆ 15378         ┆ false  │
│ Bronny James      ┆ 0.3944   ┆ 18084         ┆ false  │
│ Jimmy Butler      ┆ 0.3909   ┆ 18858         ┆ false  │
│ Zion Williamson   ┆ 0.3895   ┆ 10659         ┆ false  │
└───────────────────┴──────────┴───────────────┴────────┘

In [8]:
# §5b — the queue. One filtered pass over the v1 parquet.
_amap_values = ", ".join(
    f"('{sql_quote(k)}', '{sql_quote(c)}')" for k, c in sorted(KEY_TO_CANON.items())
)
PER_PLAYER = 5

_praise_df = duckdb.sql(f"""
    WITH amap(key, canon) AS (VALUES {_amap_values})
    SELECT s.comment_id AS id, s.body, a.canon, s.sentiment_player,
           s.confidence, s.score, s.author_flair_text
    FROM read_parquet('{V1_SENTIMENT}') s
    JOIN amap a
      ON lower(replace(coalesce(s.sentiment_player, ''), '.', '')) = a.key
    WHERE s.sentiment = 'pos'
      AND s.confidence >= 0.9
      AND length(s.body) BETWEEN 20 AND 300
      AND s.body NOT IN ('[deleted]', '[removed]')
    QUALIFY row_number() OVER (
        PARTITION BY a.canon ORDER BY md5(s.comment_id)
    ) <= {PER_PLAYER * 2}
""").pl()

praise_rows: list[dict] = []
for canon in STRUGGLING:
    df = _praise_df.filter(pl.col("canon") == canon)
    praise_rows += dedup_take(df, PER_PLAYER)

praise_queue = [
    make_candidate(
        cid=f"praise-m{i:02d}",
        text=r["body"].strip(),
        proposed="pos",
        proposed_player=r["canon"],
        category="genuine_praise",
        source="mined-v1",
        comment_id=r["id"],
        note="selected on the v1 classifier's own high-confidence pos label; "
        "reject false positives rather than relabeling them",
        context={
            "v1_confidence": r["confidence"],
            "score": r["score"],
            "author_flair_text": r["author_flair_text"],
            "sentiment_player_raw": r["sentiment_player"],
        },
    )
    for i, r in enumerate(praise_rows, start=1)
]
write_queue(MINING_DIR / "candidates_genuine_praise.yaml", praise_queue, "§5")
print(f"per player: {pl.Series([r['canon'] for r in praise_rows]).value_counts()}")

wrote 35 candidates -> data/2025-26/reference/eval_mining/candidates_genuine_praise.yaml
per player: shape: (7, 2)
┌───────────────────┬───────┐
│                   ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ Russell Westbrook ┆ 5     │
│ Jamal Murray      ┆ 5     │
│ Rudy Gobert       ┆ 5     │
│ Joel Embiid       ┆ 5     │
│ James Harden      ┆ 5     │
│ Ben Simmons       ┆ 5     │
│ Draymond Green    ┆ 5     │
└───────────────────┴───────┘


## 6. Slang mining

Three passes, all guarded: (a) v1 tokenization split by sentiment → log-odds
(which terms *lean* pos/neg in classified data); (b) v2 tokenization →
frequency-ratio vs v1 (which terms are *new* in 25-26); (c) merged ranked table
with capped example snippets → `slang_candidates.yaml` for nba-superfan
verification.

**Exclusions** (noise, not candidates): the prompt's existing slang terms,
player-name/alias tokens from both season configs, and stopwords — **minus the
§4.2 keep-list** (`him`, `dawg`, `hooper`, `bum`, `poverty`), which is
force-included with its stats since stopword filters would otherwise eat it.

In [9]:
# §6a — v1 token counts by sentiment, then log-odds (add-1 smoothing).
V1_TOK_PQ = MINING_DIR / "slang_v1_tokens.parquet"
warn_if_stale(V1_TOK_PQ, V1_SENTIMENT)

KEEP_LIST = {"him", "dawg", "hooper", "bum", "poverty"}
_keep_sql = ", ".join(f"'{t}'" for t in sorted(KEEP_LIST))
# Tokens are pure a-z runs: apostrophes split, so possessives fold into their
# stem ("ayton's" -> "ayton" + dropped "s") and contraction shards die to the
# len>=2 / stopword filters. Untracked-player surnames still surface in
# new-v2 — expected noise the nba-superfan verification pass filters.
TOKENIZE = "unnest(regexp_split_to_array(lower(body), '[^a-z]+'))"

if not V1_TOK_PQ.exists():
    duckdb.sql(f"""
        COPY (
            SELECT tok, sentiment, count(*) AS n
            FROM (
                SELECT {TOKENIZE} AS tok, sentiment
                FROM read_parquet('{V1_SENTIMENT}')
                WHERE sentiment IN ('pos', 'neg')
            )
            WHERE len(tok) >= 2
            GROUP BY tok, sentiment
            HAVING count(*) >= 50 OR tok IN ({_keep_sql})
        ) TO '{V1_TOK_PQ}' (FORMAT parquet)
    """)

_tok = (
    pl.read_parquet(V1_TOK_PQ)
    .pivot(values="n", index="tok", on="sentiment")
    .fill_null(0)
    .rename({"pos": "n_pos", "neg": "n_neg"})
    .sort("tok")  # pivot order is nondeterministic; queues must reproduce
)
N_POS, N_NEG, VOCAB = _tok["n_pos"].sum(), _tok["n_neg"].sum(), _tok.height
logodds = _tok.with_columns(
    (
        ((pl.col("n_neg") + 1) / (N_NEG + VOCAB)).log()
        - ((pl.col("n_pos") + 1) / (N_POS + VOCAB)).log()
    ).alias("log_odds")
)

# Exclusions: prompt slang (pipeline/batch.py build_prompt), player-name/alias
# tokens from both season configs, stopwords. Keep-list bypasses everything.
PROMPT_SLANG = {"nasty", "sick", "filthy", "washed", "brick", "fraud", "cooked", "goat"}

ALIAS_TOKENS: set[str] = set()
for _cfg_path in ("config/2024-25/players.yaml", "config/2025-26/players.yaml"):
    with open(_cfg_path) as f:
        for _name, _entry in yaml.safe_load(f)["players"].items():
            _aliases = _entry.get("aliases", []) if isinstance(_entry, dict) else _entry
            for _term in [_name, *_aliases]:
                ALIAS_TOKENS.update(re.split(r"[^a-z']+", _term.lower()))
ALIAS_TOKENS.discard("")

STOPWORDS = set("""
a about after again all also am an and any are as at back be because been
before being between both but by can could day did do does doesn't don't down
even first for from get go going good got had has have he her here him his how
i if in into is it its it's just know like ll me more most much my new no not
now of off on one only or other our out over re s said same see she so some
still such t than that that's the their them then there these they thing think
this those time to too two up us use very was way we well were what when where
which while who why will with would you your
""".split())

EXCLUDE = (PROMPT_SLANG | ALIAS_TOKENS | STOPWORDS) - KEEP_LIST
ranked = logodds.filter(
    (~pl.col("tok").is_in(sorted(EXCLUDE)))
    & ((pl.col("n_pos") + pl.col("n_neg") >= 200) | pl.col("tok").is_in(sorted(KEEP_LIST)))
)
ranked.write_parquet(MINING_DIR / "slang_v1_logodds.parquet")

# tok is the tie-breaker throughout: zero-count terms share log-odds values.
neg_top = ranked.sort(["log_odds", "tok"], descending=[True, False]).head(25)
pos_top = ranked.sort(["log_odds", "tok"]).head(25)
keep_rows = ranked.filter(pl.col("tok").is_in(sorted(KEEP_LIST)))
print("Top neg-leaning:", neg_top.head(10))
print("Top pos-leaning:", pos_top.head(10))
print("Keep-list stats:", keep_rows)

Top neg-leaning: shape: (10, 4)
┌───────────┬───────┬───────┬──────────┐
│ tok       ┆ n_neg ┆ n_pos ┆ log_odds │
│ ---       ┆ ---   ┆ ---   ┆ ---      │
│ str       ┆ i64   ┆ i64   ┆ f64      │
╞═══════════╪═══════╪═══════╪══════════╡
│ charmin   ┆ 579   ┆ 0     ┆ 5.973029 │
│ yikes     ┆ 528   ┆ 0     ┆ 5.88099  │
│ shameless ┆ 447   ┆ 0     ┆ 5.714795 │
│ softest   ┆ 434   ┆ 0     ┆ 5.685347 │
│ pos       ┆ 432   ┆ 0     ┆ 5.680739 │
│ assault   ┆ 429   ┆ 0     ┆ 5.673787 │
│ elbowed   ┆ 425   ┆ 0     ┆ 5.664441 │
│ phantom   ┆ 397   ┆ 0     ┆ 5.596453 │
│ injuring  ┆ 386   ┆ 0     ┆ 5.568426 │
│ bricked   ┆ 375   ┆ 0     ┆ 5.539591 │
└───────────┴───────┴───────┴──────────┘
Top pos-leaning: shape: (10, 4)
┌─────────────┬───────┬───────┬───────────┐
│ tok         ┆ n_neg ┆ n_pos ┆ log_odds  │
│ ---         ┆ ---   ┆ ---   ┆ ---       │
│ str         ┆ i64   ┆ i64   ┆ f64       │
╞═════════════╪═══════╪═══════╪═══════════╡
│ goated      ┆ 0     ┆ 309   ┆ -6.126571 │
│ lfg         ┆ 

In [10]:
# §6b — v2 token frequency (aggregate-only pass over the 1 GB file) and the
# new-slang ratio vs v1.
V2_FREQ_PQ = MINING_DIR / "slang_v2_freq.parquet"
warn_if_stale(V2_FREQ_PQ, MENTIONS_RAW)

if not V2_FREQ_PQ.exists():
    duckdb.sql(f"""
        COPY (
            SELECT tok, count(*) AS n_v2
            FROM (
                SELECT {TOKENIZE} AS tok
                FROM read_ndjson('{MENTIONS_RAW}', format='newline_delimited',
                     columns={{body: 'VARCHAR'}})
            )
            WHERE len(tok) >= 2
            GROUP BY tok
            HAVING count(*) >= 100 OR tok IN ({_keep_sql})
        ) TO '{V2_FREQ_PQ}' (FORMAT parquet)
    """)

_v2 = pl.read_parquet(V2_FREQ_PQ)
T2 = _v2["n_v2"].sum()
_v1_totals = logodds.select(
    "tok", (pl.col("n_pos") + pl.col("n_neg")).alias("n_v1")
)
T1 = _v1_totals["n_v1"].sum()

new_slang = (
    _v2.join(_v1_totals, on="tok", how="left")
    .fill_null(0)
    .with_columns(
        ((pl.col("n_v2") / T2) / ((pl.col("n_v1") + 1) / T1)).alias("ratio_v2_v1")
    )
    .filter(
        (~pl.col("tok").is_in(sorted(EXCLUDE)))
        & (pl.col("ratio_v2_v1") >= 3.0)
        & (pl.col("n_v2") >= 100)
    )
    .sort(["ratio_v2_v1", "tok"], descending=[True, False])
    .head(25)
)
print(new_slang)

shape: (25, 4)
┌─────────────┬──────┬──────┬─────────────┐
│ tok         ┆ n_v2 ┆ n_v1 ┆ ratio_v2_v1 │
│ ---         ┆ ---  ┆ ---  ┆ ---         │
│ str         ┆ i64  ┆ i64  ┆ f64         │
╞═════════════╪══════╪══════╪═════════════╡
│ ayo         ┆ 3826 ┆ 0    ┆ 1775.507985 │
│ aspiration  ┆ 2136 ┆ 0    ┆ 991.240213  │
│ laravia     ┆ 1801 ┆ 0    ┆ 835.77885   │
│ cmb         ┆ 1490 ┆ 0    ┆ 691.455018  │
│ peterson    ┆ 1473 ┆ 0    ┆ 683.565933  │
│ …           ┆ …    ┆ …    ┆ …           │
│ acuff       ┆ 579  ┆ 0    ┆ 268.692923  │
│ dosunmu     ┆ 578  ┆ 0    ┆ 268.228859  │
│ beringer    ┆ 571  ┆ 0    ┆ 264.980413  │
│ goaltending ┆ 557  ┆ 0    ┆ 258.48352   │
│ monks       ┆ 542  ┆ 0    ┆ 251.522563  │
└─────────────┴──────┴──────┴─────────────┘


In [11]:
# §6c — merged ranked table + capped snippets -> slang_candidates.yaml.
_shortlist: dict[str, dict] = {}
for df, direction in ((neg_top, "neg-leaning"), (pos_top, "pos-leaning"), (keep_rows, "keep-list")):
    for r in df.iter_rows(named=True):
        _shortlist.setdefault(r["tok"], {**r, "direction": direction, "n_v2": None, "ratio_v2_v1": None})
for r in new_slang.iter_rows(named=True):
    if r["tok"] in _shortlist:
        _shortlist[r["tok"]].update(n_v2=r["n_v2"], ratio_v2_v1=r["ratio_v2_v1"])
    else:
        _shortlist[r["tok"]] = {
            "tok": r["tok"], "n_pos": None, "n_neg": None, "log_odds": None,
            "direction": "new-v2", "n_v2": r["n_v2"], "ratio_v2_v1": r["ratio_v2_v1"],
        }

# Example snippets: one pass over v1 bodies for all shortlist terms; new-v2
# terms missing from v1 get their snippets from the (small) §1 pool parquet
# where possible, else none.
_terms_values = ", ".join(f"('{sql_quote(t)}')" for t in sorted(_shortlist))
_snips = duckdb.sql(f"""
    WITH terms(term) AS (VALUES {_terms_values})
    SELECT t.term,
           substr(s.body, greatest(1, strpos(lower(s.body), t.term) - 40), 120) AS snip
    FROM read_parquet('{V1_SENTIMENT}') s
    JOIN terms t ON strpos(lower(s.body), t.term) > 0
    QUALIFY row_number() OVER (PARTITION BY t.term ORDER BY md5(s.comment_id)) <= 2
    ORDER BY 1, 2
""").pl()
_snip_map: dict[str, list[str]] = {}
for r in _snips.iter_rows(named=True):
    _snip_map.setdefault(r["term"], []).append(r["snip"])

_missing = sorted(t for t in _shortlist if t not in _snip_map)
if _missing:
    _mv = ", ".join(f"('{sql_quote(t)}')" for t in _missing)
    _pool_snips = duckdb.sql(f"""
        WITH terms(term) AS (VALUES {_mv})
        SELECT t.term,
               substr(p.body, greatest(1, strpos(lower(p.body), t.term) - 40), 120) AS snip
        FROM read_parquet('{POOL_PQ}') p
        JOIN terms t ON strpos(lower(p.body), t.term) > 0
        QUALIFY row_number() OVER (PARTITION BY t.term ORDER BY md5(p.id)) <= 2
        ORDER BY 1, 2
    """).pl()
    for r in _pool_snips.iter_rows(named=True):
        _snip_map.setdefault(r["term"], []).append(r["snip"])

slang_rows = [
    {
        "term": v["tok"],
        "direction": v["direction"],
        "log_odds": round(v["log_odds"], 3) if v["log_odds"] is not None else None,
        "ratio_v2_v1": round(v["ratio_v2_v1"], 1) if v["ratio_v2_v1"] is not None else None,
        "n_pos_v1": v["n_pos"],
        "n_neg_v1": v["n_neg"],
        "n_v2": v["n_v2"],
        "in_keep_list": v["tok"] in KEEP_LIST,
        "examples": _snip_map.get(v["tok"], []),
    }
    for v in sorted(
        _shortlist.values(),
        key=lambda v: (
            v["direction"],
            -(v["log_odds"] or v["ratio_v2_v1"] or 0),
            v["tok"],
        ),
    )
]

SLANG_YAML = MINING_DIR / "slang_candidates.yaml"
_slang_header = (
    "# slang_candidates.yaml — written by notebooks/2025-26/05_case_mining.ipynb (§6)\n"
    "#\n"
    "# Ranked slang candidates for nba-superfan verification (#62 item 2).\n"
    "# direction: neg-leaning / pos-leaning (v1 log-odds), new-v2 (frequency\n"
    "# ratio vs v1), keep-list (§4.2 shortlist, force-included).\n"
    "# VERIFICATION: is the term actually sentiment-bearing NBA slang (not an\n"
    "# artifact)? Verified terms feed the prompt experiment PR and get eval\n"
    "# suite cases; nothing here changes the prompt directly.\n"
    "#\n"
)
SLANG_YAML.write_text(
    _slang_header
    + yaml.safe_dump({"candidates": slang_rows}, sort_keys=False, allow_unicode=True, width=4096)
)
print(f"wrote {len(slang_rows)} slang candidates -> {SLANG_YAML}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote 80 slang candidates -> data/2025-26/reference/eval_mining/slang_candidates.yaml


## 7. Verdict & handoff

Queue inventory, determinism hashes (re-running against the frozen inputs must
reproduce these), and collision re-checks against `tests/eval/cases.yaml`.

In [12]:
# §7 — inventory, hashes, collision asserts.
QUEUES = [
    MINING_DIR / "candidates_short.yaml",
    MINING_DIR / "candidates_multi_player.yaml",
    MINING_DIR / "candidates_sarcasm.yaml",
    MINING_DIR / "candidates_genuine_praise.yaml",
    MINING_DIR / "slang_candidates.yaml",
]

_case_texts = {c.text.strip().lower() for c in load_cases()}
_case_ids = {c.comment_id for c in load_cases() if c.comment_id}

for q in QUEUES:
    payload = yaml.safe_load(q.read_text())["candidates"]
    digest = hashlib.sha256(q.read_bytes()).hexdigest()[:16]
    print(f"{q.name:38} {len(payload):3} candidates  sha256:{digest}")
    if q.name.startswith("candidates_"):
        for c in payload:
            assert c["text"].strip().lower() not in _case_texts, (
                f"{c['id']} collides with an existing case text"
            )
            assert c["comment_id"] not in _case_ids, (
                f"{c['id']} comment_id already used in cases.yaml"
            )
print("\nNo collisions with tests/eval/cases.yaml.")

candidates_short.yaml                   30 candidates  sha256:dd96ae501aed9579
candidates_multi_player.yaml            30 candidates  sha256:636b55fbf4a487cb
candidates_sarcasm.yaml                 30 candidates  sha256:917b5b580e5d6034
candidates_genuine_praise.yaml          35 candidates  sha256:2bc76c4e10b27963
slang_candidates.yaml                   80 candidates  sha256:d7788f56baf94f3b

No collisions with tests/eval/cases.yaml.


### Handoff checklist

1. **Label** each `candidates_*.yaml` (instructions in each file header):
   fix `proposed`, rename to `expected`/`expected_player`, delete rejects and
   `context:` blocks.
2. **Verify** with the nba-superfan agent (Claude Code, outside this notebook):
   slang candidates in `slang_candidates.yaml`, plus a pass over the labeled
   queues for mislabeled/ambiguous entries.
3. **Merge** survivors into `tests/eval/cases.yaml`: new categories `short`,
   `multi_player`, `genuine_praise` need `meta.category_floors` entries
   (placeholder `0.0`); `/s` cases join the existing `sarcasm` category.
4. **Baseline**: `uv run pytest -m eval --maxfail=0 -rxX` × 3 runs; mark stable
   misses `known_miss: true`; pin new floors (baseline minus one case of slack
   for categories with ≥ 5 cases); re-pin `sarcasm`.
5. `uv run pytest` (offline) to confirm nothing else moved.

**No prompt change happens in this notebook or its PR.** Slang-line and
sarcasm-line decisions are the experiment PR's job, measured against the
expanded suite this feeds.